# Burn cost workflow demo: optional test of the weekly runner

Use this notebook when configuring or debugging `monitoring.py`. Scheduled runs
execute that Python file directly. This notebook is not needed each week.

Running the test executes the real workflow against the configured database.
It scores the champion and saves three challengers, recording four monitoring
observations. It writes SQL results; it is not a dry run.

| Variant | Role and result |
| --- | --- |
| `STATIC_SCORE` | Score the current champion with its deployed coefficients and fitted settings. |
| `FROZEN_REFIT` | Publish a challenger with refitted coefficients and the saved basis and penalties. |
| `REESTIMATE_LAMBDA` | Publish a challenger with refitted coefficients and penalties using the saved basis. |
| `FULL_ADAPTIVE` | Publish a challenger after rebuilding the basis and refitting coefficients and penalties. |

This run does not promote a challenger. Review and explicitly promote a selected
package in `06_model_deployment.ipynb`.


## Publishing a new recipe does not change this baseline

03 saves a challenger with the corresponding model version. It stays a challenger
until you explicitly promote it in 06. This runner always uses the champion in
its configured `DEPLOYMENT_SLOT`; it does not automatically pick the newest package.
Testing a selected undeployed challenger is not yet an option in this notebook.

When preparing a new feature for recurring runs, include it in `monitoring.py`'s
source query as well as 01's query. The runner reads fresh data independently of 01.


## Check the weekly loader

`monitoring.py` is already filled in for this demo. It reads the latest snapshot from `dbo.DEMO_BURN_COST_SOURCE`, including its source date.
The first run uses 15 September, while the champion was fitted on 31 August.
It loads the champion definition and fitted state from SQL; it does not read `prototype.toml` or the dataset saved by 01.

Notebook 07 and scheduled runs call the same `run()` function. This cell reloads the module so edits take effect.


In [ ]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "pricing_models").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pricing_models.burn_cost_demo import monitoring

monitoring = importlib.reload(monitoring)
print(f"Configuration: {monitoring.__file__}")

## Test one weekly run

This writes four monitoring observations and three challenger packages to the
configured database. Confirm its destination and enable the remote write guard in
`monitoring.py` before an authorized run. Each observation is saved separately.
If a later operation fails, earlier observations and publications can remain;
repeating the same evidence reuses prior results.

The runs table shows the champion scored by this run and the three saved challengers.
`Model version` stays the same across these weekly refits. `Package` is the saved result to
select in notebook 06. `Compared with package` identifies the champion used for this run.
The static score uses the champion's existing package; it does not create another one.

Compare metrics, warnings and categorical drift before choosing a challenger for
review in 06. Exceptions keep their Python traceback in this notebook.

The full SQL identifiers remain in `report.runs`. Review all saved packages and their
current roles in `pricing.V_MODEL_REGISTRY`; filter `deployment_slot` to match this model.


In [ ]:
report = monitoring.run()
display({"manifest_id": report.manifest_id})
print("Champion at run time and saved challengers")
runs = report.runs.loc[
    :,
    [
        "variant",
        "role",
        "definition_revision",
        "package_version",
        "baseline_package_version",
        "publication_reused",
    ],
].copy()
# Scoring the champion uses its existing package; it does not create another one.
static_score = runs["variant"].eq("STATIC_SCORE")
runs.loc[static_score, "package_version"] = runs.loc[static_score, "baseline_package_version"]
display(
    runs.rename(
        columns={
            "variant": "Fit",
            "role": "Role",
            "definition_revision": "Model version",
            "package_version": "Package",
            "baseline_package_version": "Compared with package",
            "publication_reused": "Reused saved result?",
        }
    )
)
print("Metrics")
display(report.metrics)
print("Warnings and data checks")
display(report.issues)
print("Categorical drift")
display(report.drift)

## Run the same file on a schedule

The next cell prints the exact Python executable and file command for this project.
Use the project environment's kernel, then run the printed command once in a terminal
and review its log before scheduling it.

For Windows Task Scheduler, use the printed Python executable as **Program/script**
and the quoted `monitoring.py` path as **Add arguments**. For WSL cron, use the printed
command as the job command. Both can run from any working directory. Keep the project
path and interpreter fixed. The machine, WSL when used, source system, and database
must be available at run time.

Each script invocation prints its log path under `.local/monitoring_logs` and returns
a nonzero exit code on failure. This notebook does not register a scheduler task.


In [ ]:
import os
import shlex
import subprocess

script_path = Path(monitoring.__file__).resolve()
command = [sys.executable, str(script_path)]
print(subprocess.list2cmdline(command) if os.name == "nt" else shlex.join(command))
print(f"Python executable: {sys.executable}")
print(f'File argument: "{script_path}"')
print(f"Logs: {monitoring.MODEL_DIR / '.local' / 'monitoring_logs'}")

Next: **08_inspect_sql.ipynb** shows the champion, challengers, metrics and compressed recipe. Return to **06** only if you decide to promote a challenger.